# Example 17 — User-friendly simultaneous CP fit

This notebook shows the concise `CPFitSession` workflow for $B^\pm\to K^\pm\pi^+\pi^-$. It keeps the same joint charge-Dalitz normalization used by `CPJointNLL`, but removes the manual cache/background/NLL/minimizer assembly.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    CPBackgroundSpec, CPFitSession, CPRealImag, DecayChannel, DecayModel,
    NonResonant, Parameter, PhaseSpaceSample, Resonance, enable_x64,
    plot_dalitz, weighted_resample,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


## 1. Shared CP amplitude parameters


In [ ]:
truth_spec = {
    'Kstar892': (1.00, 0.00, +0.04, -0.03),
    'rho770':   (0.65, 0.10, +0.06, +0.04),
    'NR':       (-0.50, 0.10, 0.00, 0.00),
}
truth, shared = {}, {}
for name, (x, y, dx, dy) in truth_spec.items():
    pars = (
        Parameter.coefficient(f'{name}.x', x, owner=name, fixed=(name == 'Kstar892'), step=0.01),
        Parameter.coefficient(f'{name}.y', y, owner=name, fixed=(name == 'Kstar892'), step=0.01),
        Parameter.coefficient(f'{name}.dx', dx, owner=name, fixed=(name == 'NR'), step=0.01),
        Parameter.coefficient(f'{name}.dy', dy, owner=name, fixed=(name == 'NR'), step=0.01),
    )
    shared[name] = CPRealImag(*pars)
    truth.update({p.name: p.value for p in pars})

def components(charge):
    c = {name: coeff.for_charge(charge) for name, coeff in shared.items()}
    return [
        Resonance('Kstar892', (0,2), c['Kstar892'], mass=0.8958, width=0.0474, spin=1),
        Resonance('rho770', (1,2), c['rho770'], mass=0.7753, width=0.1491, spin=1),
        NonResonant(c['NR']),
    ]

plus_model = DecayModel(
    DecayChannel('B+', ('K+','pi+','pi-')), components(+1),
    normalization_method='square-dalitz', normalization_resolution=250, normalization_pair=(0,2),
)
minus_model = DecayModel(
    DecayChannel('B-', ('K-','pi-','pi+')), components(-1),
    normalization_method='square-dalitz', normalization_resolution=250, normalization_pair=(0,2),
)


## 2. Efficiency, background and toy generation


In [ ]:
mK, mpi, _ = plus_model.channel.daughter_masses
mB = plus_model.channel.parent_mass
s13_min, s13_max = (mK+mpi)**2, (mB-mpi)**2
s23_min, s23_max = (2*mpi)**2, (mB-mK)**2

def scaled(data, key, low, high):
    return jnp.clip((data[key]-low)/(high-low), 0.0, 1.0)

efficiency = FunctionalEfficiency(
    lambda d: 0.60 + 0.25*scaled(d,'s13',s13_min,s13_max)
)
background = FunctionalBackground(
    lambda d: 0.5 + 1.0*scaled(d,'s13',s13_min,s13_max) + 0.3*scaled(d,'s23',s23_min,s23_max)
)

N_POOL, N_DATA, F_SIG_TRUE = 100_000, 20_000, 0.82
plus_pool = plus_model.generate_phase_space(N_POOL, seed=17001)
minus_pool = minus_model.generate_phase_space(N_POOL, seed=17002)
plus_norm, minus_norm = plus_model.normalization_sample, minus_model.normalization_sample
plus_cache = plus_model.prepare_cache(plus_pool, plus_norm, efficiency_normalization=efficiency(plus_norm.as_dict()))
minus_cache = minus_model.prepare_cache(minus_pool, minus_norm, efficiency_normalization=efficiency(minus_norm.as_dict()))
i_plus = float(plus_cache.normalization(truth)); i_minus = float(minus_cache.normalization(truth))
p_plus = i_plus/(i_plus+i_minus)
rng = np.random.default_rng(17003)
n_sig = rng.binomial(N_DATA, F_SIG_TRUE); n_bkg = N_DATA-n_sig
n_sig_plus = rng.binomial(n_sig, p_plus); n_sig_minus = n_sig-n_sig_plus
n_bkg_plus = rng.binomial(n_bkg, 0.5); n_bkg_minus = n_bkg-n_bkg_plus
plus_signal = weighted_resample(jax.random.key(17004), plus_pool, plus_pool.weights*efficiency(plus_pool.as_dict())*plus_cache.intensity(truth), n_sig_plus)
minus_signal = weighted_resample(jax.random.key(17005), minus_pool, minus_pool.weights*efficiency(minus_pool.as_dict())*minus_cache.intensity(truth), n_sig_minus)
plus_bkg = weighted_resample(jax.random.key(17006), plus_pool, plus_pool.weights*background(plus_pool.as_dict()), n_bkg_plus)
minus_bkg = weighted_resample(jax.random.key(17007), minus_pool, minus_pool.weights*background(minus_pool.as_dict()), n_bkg_minus)

def merge(a,b):
    return PhaseSpaceSample(
        s12=jnp.concatenate((a.s12,b.s12)), s13=jnp.concatenate((a.s13,b.s13)),
        s23=jnp.concatenate((a.s23,b.s23)), weights=jnp.ones(a.size+b.size),
    )

plus_data, minus_data = merge(plus_signal, plus_bkg), merge(minus_signal, minus_bkg)
fig, axes = plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
plot_dalitz(plus_data, ax=axes[0], title='B+ toy', colorbar=False)
plot_dalitz(minus_data, ax=axes[1], title='B- toy', colorbar=False)
plt.show()


## 3. Build and run the fit in a few lines

`CPFitSession` automatically builds the two prepared caches, folds the efficiency into normalization, normalizes the background jointly over charge, constructs `CPJointNLL`, collects shared parameters and creates the minimizer.


In [ ]:
f_sig = Parameter('signal_fraction', 0.74, bounds=(0.50,0.99), step=0.01)
session = CPFitSession(
    plus_model, minus_model, plus_data, minus_data,
    plus_efficiency=efficiency, minus_efficiency=efficiency,
    backgrounds=(CPBackgroundSpec('combinatorial', background),),
    signal_fraction=f_sig,
)

rng = np.random.default_rng(17008)
start = {p.name: float(p.value + rng.normal(0,0.07)) for p in session.parameters if not p.fixed}
start['signal_fraction'] = 0.74
result = session.fit(start, simplex=True, ncall=40_000, verbose=1)


## 4. One-call report


In [ ]:
report = session.report(
    result,
    include_fit_fractions=True,
    acceptance_weighted_fractions=True,
)
print('report keys:', report.keys())


## 5. Automatic charge-separated projections

The two charge projections use one common global normalization. They therefore retain the predicted integrated charge asymmetry instead of independently normalizing B+ and B-.


In [ ]:
session.plot_projection(result, 's13', bins=55)
plt.show()
session.plot_projection(result, 's23', bins=55)
plt.show()


## 6. Minimal ROOT usage

For real data the same session can be created directly from two TTrees:

```python
session = CPFitSession.from_root(
    plus_model, minus_model,
    'Bplus.root', 'DecayTree',
    'Bminus.root', 'DecayTree',
    plus_root_kwargs={'s12':'S12','s13':'S13','s23':'S23'},
    minus_root_kwargs={'s12':'S12','s13':'S13','s23':'S23'},
)
result = session.fit()
session.report(result)
session.plot_projection(result, 's13')
```
